# Resource-grid raw-data viewer

This notebook reads a completed `resource_grid.csv`. It does not run resource estimators. Set `RESOURCE_GRID_DATA` to inspect a non-default output.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
DATA_PATH = Path(os.environ.get(
    'RESOURCE_GRID_DATA',
    root / 'outputs' / 'resource_grid' / 'full' / 'resource_grid.csv',
))
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} does not exist. Run hamiltonian-resource-grid first.'
    )
data = pd.read_csv(DATA_PATH)
print(f'Loaded {len(data):,} rows from {DATA_PATH}')

In [ ]:
status_summary = (
    data.groupby(['hamiltonian_model', 'estimator_variant', 'status'])
    .size()
    .rename('rows')
    .reset_index()
)
display(status_summary)

## Viewer selections

Change these lightweight selectors and rerun the plotting cells.

In [ ]:
MODEL = 'transverse_field_ising'
METRIC = 't_count'  # or cnot_count
SELECTED_LOG_ERRORS = (-1.0, -2.0, -3.0, -4.0)
SELECTED_SIZES = (3, 20, 60, 120)
HEATMAP_METHOD = 'trotter-p4'

ok = data[data['status'].eq('ok')].copy()
model_data = ok[ok['hamiltonian_model'].eq(MODEL)]

In [ ]:
figure, axes = plt.subplots(1, len(SELECTED_LOG_ERRORS), figsize=(16, 4), sharey=True)
for axis, log_error in zip(np.atleast_1d(axes), SELECTED_LOG_ERRORS):
    selected = model_data[np.isclose(model_data['log10_target_error'], log_error)]
    for method, group in selected.groupby('method_id', sort=False):
        axis.loglog(group['system_qubits'], group[METRIC], label=method)
    axis.set(title=f'log10(error)={log_error:g}', xlabel='N')
    axis.grid(True, which='both', alpha=0.25)
axes[0].set_ylabel(METRIC)
axes[-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
figure.suptitle(f'{MODEL}: {METRIC} versus system size')
figure.tight_layout()

In [ ]:
available_sizes = set(model_data['system_qubits'].astype(int))
sizes = [size for size in SELECTED_SIZES if size in available_sizes]
figure, axes = plt.subplots(1, len(sizes), figsize=(4 * len(sizes), 4), sharey=True)
for axis, size in zip(np.atleast_1d(axes), sizes):
    selected = model_data[model_data['system_qubits'].eq(size)]
    for method, group in selected.groupby('method_id', sort=False):
        group = group.sort_values('target_error')
        axis.loglog(group['target_error'], group[METRIC], label=method)
    axis.invert_xaxis()
    axis.set(title=f'N={size}', xlabel='target error')
    axis.grid(True, which='both', alpha=0.25)
axes[0].set_ylabel(METRIC)
axes[-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
figure.suptitle(f'{MODEL}: {METRIC} versus target error')
figure.tight_layout()

In [ ]:
heatmap_rows = model_data[model_data['method_id'].eq(HEATMAP_METHOD)]
heatmap = heatmap_rows.pivot(
    index='log10_target_error', columns='system_qubits', values=METRIC
).sort_index(ascending=False)
figure, axis = plt.subplots(figsize=(12, 5))
image = axis.imshow(np.log10(heatmap), aspect='auto', origin='upper')
axis.set(
    title=f'{MODEL}: log10({METRIC}), {HEATMAP_METHOD}',
    xlabel='N column index', ylabel='log10(target error) row index',
)
figure.colorbar(image, ax=axis, label=f'log10({METRIC})')
figure.tight_layout()

In [ ]:
comparison = model_data[
    model_data['estimator_variant'].isin(['empirical', 'commutator'])
    & np.isclose(model_data['log10_target_error'], -2.0)
]
figure, axis = plt.subplots(figsize=(10, 5))
for (family, variant), group in comparison.groupby(
    ['method_family', 'estimator_variant'], sort=False
):
    axis.loglog(
        group['system_qubits'], group[METRIC],
        marker='o', markersize=3, label=f'{family} {variant}',
    )
axis.set(xlabel='N', ylabel=METRIC, title='Empirical/commutator comparison')
axis.grid(True, which='both', alpha=0.25)
axis.legend()
figure.tight_layout()

In [ ]:
selected = model_data[np.isclose(model_data['log10_target_error'], -2.0)]
figure, axes = plt.subplots(1, 4, figsize=(16, 4))
parameter_specs = (
    ('trotter_reps', 'Trotter repetitions r', 'trotter'),
    ('mpf_branch_count', 'MPF branch count J', 'multiproduct'),
    ('mpf_segments', 'MPF segment count', 'multiproduct'),
    ('qsvt_degree', 'QSVT degree d', 'qsvt'),
)
for axis, (column, title, family) in zip(axes, parameter_specs):
    rows = selected[selected['method_family'].eq(family)]
    for method, group in rows.groupby('method_id', sort=False):
        axis.plot(group['system_qubits'], group[column], label=method)
    axis.set(title=title, xlabel='N')
    axis.grid(True, alpha=0.25)
    if not rows.empty:
        axis.legend(fontsize=6)
figure.tight_layout()

In [ ]:
missing = data[
    data['hamiltonian_model'].eq(MODEL)
    & data['estimator_variant'].eq('empirical')
].copy()
missing['is_missing'] = missing['status'].eq('missing_empirical')
mask = missing.pivot_table(
    index='log10_target_error', columns='system_qubits',
    values='is_missing', aggfunc='max', fill_value=False,
).sort_index(ascending=False)
figure, axis = plt.subplots(figsize=(12, 4))
image = axis.imshow(mask.astype(int), aspect='auto', cmap='Greys', vmin=0, vmax=1)
axis.set(
    title=f'{MODEL}: missing empirical calibration mask',
    xlabel='N column index', ylabel='log10(target error) row index',
)
figure.colorbar(image, ax=axis, ticks=(0, 1), label='missing')
figure.tight_layout()

In [ ]:
fallback_summary = (
    data[data['mpf_locality_fallback'].fillna(False).astype(bool)]
    .groupby([
        'hamiltonian_model', 'method_id', 'mpf_branch_count',
        'mpf_locality_fallback_reason',
    ], dropna=False)
    .size()
    .rename('rows')
    .reset_index()
)
display(fallback_summary)